# UAE bank SMS → clean transaction dataset

This notebook turns the censored bank-SMS Excel exports into one clean, tested transaction table.

**What it does**
1. Loads every Excel file listed in the config (any number of sheets per file).
2. Parses each SMS with bank-specific rules (Liv, Mashreq, ADCB, ADIB) into a transaction type, amount, currency, card, merchant and balance.
3. Cleans the data: splits merchant from location, normalises merchant names, converts pegged currencies to AED, flags duplicates, buy-now-pay-later and money-transfer merchants, and applies the date cutoff.
4. Runs tests: parser unit tests, data-quality checks and a privacy scan. The run stops if a hard test fails.
5. Exports the **personal** dataset and the **Khalifa University** dataset to **separate** files. They are never merged.

**How to run:** put the Excel files in a `data/` folder next to this notebook (or change `DATA_DIR`), then *Run All*.

**Expense definition:** `is_expense = True` for card purchases and bill payments, excluding duplicates. Refunds, cash withdrawals and bank fees are kept but flagged `is_borderline`. Transfers, internal moves, income, card repayments, OTPs, declines and notices are kept in the full table but are not expenses.

## 1. Config

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')

# Rows dated after this are dropped from the personal dataset (the Khalifa dataset ends in Jul 2023).
CUTOFF = pd.Timestamp('2025-12-31 23:59:59')

# One entry per file. dataset decides which output it goes to; user=None means "read the User column".
FILES = [
    dict(file='danish-adcb-redacted-2025.xlsx',    dataset='personal', user='danish', bank='ADCB'),
    dict(file='danish-liv-redacted-2025.xlsx',     dataset='personal', user='danish', bank='Liv'),
    dict(file='contributor-a-mashreq-redacted-2025.xlsx', dataset='personal', user='contributor_a', bank='Mashreq'),
    dict(file='contributor-b-liv-redacted-2025.xlsx', dataset='personal', user='contributor_b', bank='Liv'),
    dict(file='contributor-c-liv-redacted-2025.xlsx', dataset='personal', user='contributor_c', bank='Liv'),
    dict(file='contributor-d-fab-redacted-2025.xlsx', dataset='personal', user='contributor_d', bank='FAB'),
    dict(file='khalifa-adib-sms-redacted.xlsx',    dataset='khalifa',  user=None,     bank='ADIB'),
]

# Optional: replace user names with neutral IDs in the exports (e.g. {'danish': 'P1'}). Empty = keep as is.
USER_ALIASES = {}

# Names that must never appear in any output (add anyone you know is in the raw exports).
# The real list is private: put it in private/config.py (not in the repo), e.g. BANNED_TERMS = ['FIRSTNAME LASTNAME', ...]
BANNED_TERMS = []

# Local overrides kept out of the public repo: private/config.py may redefine FILES and BANNED_TERMS
# (for example with the original file names of your own exports).
try:
    exec(open('private/config.py').read())
except FileNotFoundError:
    pass

# Fixed AED rates for currencies pegged to the US dollar. Anything else gets amount_aed = NaN.
AED_PER_UNIT = {'AED': 1.0, 'USD': 3.6725, 'SAR': 3.6725 / 3.75, 'QAR': 3.6725 / 3.64,
                'BHD': 3.6725 / 0.376, 'OMR': 3.6725 / 0.3845}

## 2. Imports and the test helper

In [ ]:
import re
import numpy as np

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

TEST_RESULTS = []

def check(name, passed, detail='', hard=True):
    # Record one test. hard=True tests stop the run at the end if they fail; soft ones only warn.
    TEST_RESULTS.append(dict(test=name, passed=bool(passed), hard=hard, detail=str(detail)[:300]))
    flag = 'PASS' if passed else ('FAIL' if hard else 'WARN')
    print(f'[{flag}] {name}' + (f'  ->  {detail}' if detail and not passed else ''))

## 3. Load the Excel files

In [ ]:
def load_file(spec):
    path = DATA_DIR / spec['file']
    sheets = pd.read_excel(path, sheet_name=None)
    frames = []
    for sheet_name, df in sheets.items():
        date_col = next(c for c in df.columns if str(c).lower().startswith('date'))
        msg_col = next(c for c in df.columns if str(c).lower() in ('message', 'sms'))
        out = pd.DataFrame({
            'date': pd.to_datetime(df[date_col]),
            'message': df[msg_col].astype(str).str.strip(),
        })
        out['user'] = df['User'].astype(str) if spec['user'] is None else spec['user']
        out['bank'] = spec['bank']
        out['dataset'] = spec['dataset']
        out['source_file'] = spec['file']
        out['source_sheet'] = sheet_name
        out['source_row'] = np.arange(2, len(df) + 2)   # Excel row number, header = row 1
        frames.append(out)
    return pd.concat(frames, ignore_index=True)

missing = [f['file'] for f in FILES if not (DATA_DIR / f['file']).exists()]
assert not missing, f'Missing input files in {DATA_DIR.resolve()}: {missing}'

raw = pd.concat([load_file(f) for f in FILES], ignore_index=True)
raw = raw[raw.message.ne('') & raw.message.ne('nan')].reset_index(drop=True)
print(f'{len(raw):,} messages loaded')
raw.groupby(['dataset', 'user', 'bank']).agg(messages=('message', 'size'),
                                             first=('date', 'min'), last=('date', 'max'))

## 4. Parser rules

Rules are checked in order and the first match wins: common rules first (OTPs, declines), then the bank's own rules, then a catch-all `notice`. Each rule captures named groups: `amount`, `currency`, `card`, `merchant_raw` and `balance`.

In [ ]:
AMT = r'(?P<amount>\d[\d,]*(?:\.\d+)?)'
CUR = r'(?P<currency>[A-Z]{3})'

def R(txn_type, pattern):
    return (txn_type, re.compile(pattern, re.IGNORECASE | re.DOTALL))

COMMON_RULES = [
    R('otp', r'\bOTP\b|one[- ]time[- ]password|activation key|authentication code|verification code|'
             r'use code: ?\d|security code|activation code|to complete your transaction at'),
    R('declined', r'declined|could not be completed|could not be processed|insufficient (?:funds|balance|limit)|'
                  r'has been rejected|REJECTED due to|CVV entered .* was incorrect|login to .* has failed'),
]

BANK_RULES = {
    'Liv': [
        R('purchase', rf'^\s*Purchase of {CUR} {AMT} with (?:Debit|Credit) Card ending (?P<card>\d+) at '
                      r'(?P<merchant_raw>.+?)\.\s*Avl Bal(?:ance)? is (?:AED )?(?P<balance>-?[\d,]*\.?\d*)'),
        R('purchase', rf'^\s*Purchase of {CUR} {AMT} with (?:Debit|Credit) Card ending (?P<card>\d+) at (?P<merchant_raw>[^.]+)'),   # SMS cut off before the balance
        R('refund',   rf'^\s*Purchase amount of {CUR} {AMT} at (?P<merchant_raw>.+?) on your (?:Debit|Credit) Card '
                      r'ending (?P<card>\d+) has been refunded'),
        R('refund',   rf'credit refund of amount of {CUR} {AMT} is notified by merchant (?P<merchant_raw>[^,]+)'),
        R('refund',   rf'^\s*{CUR} {AMT} has been credited to your account no\. \S+ (?!SOCIAL|IPI|TT REF|TOP UP|SB REF|'
                      r'RULE|TO LIV|Closure|to Safe|WITHDRAW|OBP|\*)(?P<merchant_raw>[A-Za-z][^:.]*?) ?[A-Za-z ]*:[A-Z]{2}$'),
        R('cash_withdrawal', rf'^\s*Cash Withdrawal of {CUR} {AMT} with Debit Card ending (?P<card>\d+) at (?P<merchant_raw>.*?)\. Avl'),
        R('bill_payment', rf'^\s*{CUR} {AMT} has been debited from your account no\. \S+ (?P<merchant_raw>SEWA|ETISALAT|DEWA|DU|FEWA|ADDC) NO\.'),
        R('bank_fee', rf'^\s*{CUR} {AMT} has been debited .*?(?P<merchant_raw>Account Maintenance Fees|Value Added Tax|Debit Card fee|Final Recov)'),
        R('internal_out', rf'^\s*{CUR} {AMT} has been debited .*?(?:to Safe with|RULE TRANSFER TO|TO LIV FROM|Dr Operative A/c For Closure)'),
        R('internal_in',  rf'^\s*{CUR} {AMT} has been credited .*?(?:to Safe with|RULE TRANSFER TO|TO LIV FROM|FROM SAFE|Closure Proceeds)'),
        R('income', rf'^\s*Interest of {CUR} {AMT} has been credited'),
        R('income', rf'^\s*Salary of {CUR} {AMT} has been credited'),
        R('refund', rf'Cash withdrawal of {CUR} {AMT} at .*? has been refunded'),
        R('transfer_out', rf'^\s*{CUR} {AMT} has been debited .*?(?:SOCIAL TRANSFER|LOCAL TRANSFER|MOBILE BANKING TRANSFER|TOP UP)'),
        R('transfer_out', rf'{CUR} {AMT} has been deducted from your account.*?(?:Telegraphic Transfer|DirectRemit|Fund Transfer)'),
        R('transfer_out', rf'{CUR} {AMT} has been debited from your (?:account|Saving Bank Account).*?(?:instant transfer|FastPay transfer)'),
        R('transfer_out', rf'Funds Transfer of {CUR} {AMT} has been made using your Debit Card ending (?P<card>\d+) at (?P<merchant_raw>[^.]+)'),
        R('transfer_in', rf'{CUR} {AMT} has been (?:credited|deposited|transferred) to (?:your )?(?:account|a/c|Saving Bank Account)'),
        R('other_debit', rf'^\s*{CUR} {AMT} has been debited'),
    ],
    'Mashreq': [
        R('purchase', rf'^\s*Thank you for using .*?Card ending (?P<card>\d+) for {CUR} {AMT} at (?P<merchant_raw>.+?) on '
                      r'\d{2}-[A-Z]{3}-\d{4}(?:.*?Available Balance is AED (?P<balance>-?[\d,]*\.?\d*))?'),
        R('purchase', rf'^\s*Mashreq (?:Debit|Credit) Card ending (?P<card>\d+) was used for a transaction of {CUR} {AMT} at '
                      r'(?P<merchant_raw>.+?) on \d'),
        R('cash_withdrawal', rf'Card ending with (?P<card>\d+) was used for cash withdrawal of {CUR} {AMT} at (?P<merchant_raw>.+?) on \d'
                             r'.*?Avl bal: AED (?P<balance>-?[\d,]*\.?\d*)'),
        R('refund', rf'credited with {CUR} {AMT} as Visa Refund'),
        R('bank_fee', rf'debited with {CUR} {AMT} (?:for|due to) (?P<merchant_raw>Fall Below Fees|breach of minimum balance|Statement Charges)'),
        R('bill_payment', rf'debited with {CUR} {AMT} for (?P<merchant_raw>Etisalat|DEWA|du) bill payment'),
        R('transfer_out', rf'(?:debited with|Amount of|fund transfer of) {CUR} {AMT}.*?(?:Aani|Instant Payments|Funds Transfer|Account to Account|excl\. applicable charges)'),
        R('income', rf'credited with {CUR} {AMT} (?:for Salary|as Joining Bonus)'),
        R('transfer_in', rf'{CUR} {AMT} has been (?:deposited|credited)'),
        R('transfer_in', rf'credited with {CUR} {AMT}'),
        R('other_debit', rf'debited with {CUR} {AMT}'),
    ],
    'ADCB': [
        R('purchase', rf'(?:Cr\.Card|Credit Card) \*+ ?(?P<card>\d+) (?:was )?used for {CUR} ?{AMT}(?: \([^)]*\))? on '
                      r'\d{2}/\d{2}/\d{4} [\d:]+ at (?P<merchant_raw>.+?)\. (?:Avl\.|Available) (?:Cr\.limit|credit limit)'
                      r'(?: is)?(?: now)? AED ?(?P<balance>-?[\d,]*\.?\d*)'),
        R('refund', rf'An amount of {CUR} ?{AMT} has been reversed to your Credit Card \*+(?P<card>\d+) on .*? by (?P<merchant_raw>.+?)\.?$'),
        R('bill_payment', rf'^\s*{CUR} ?{AMT} debited from Acc/Cr\.Card \*+(?P<card>\d+) for (?P<merchant_raw>.+?) on \d'),
        R('card_payment_received', rf'Your payment of {CUR} ?{AMT}.*?against Credit Card'),
        R('loan_repayment', rf'Installment Payment Plan .*? of {CUR} ?{AMT}'),
    ],
    'FAB': [
        R('notice', r'Credit Card Purchase\s+Card No \S+\s+[A-Z]{3} 0?\.00\s'),       # zero-amount card checks (Visa provisioning)
        R('purchase', rf'Credit Card Purchase\s+Card No X+(?P<card>\d+)\s+{CUR} {AMT}\s+(?P<merchant_raw>.+?)\s+'
                      r'\d{2}/\d{2}/\d{2} [\d:]+\s+Available Balance AED (?P<balance>-?\d[\d,]*\.?\d*)'),
        R('cash_withdrawal', rf'Credit Card Cash Withdrawal\s+Card No X+(?P<card>\d+)\s+{CUR} {AMT}\s+(?P<merchant_raw>.+?)\s+\d{{2}}/'),
        R('cash_withdrawal', rf'ATM Cash withdrawal.*?Card X+(?P<card>\d+)\s+{CUR} {AMT}'),
        R('bill_payment', rf'payment instructions of {CUR} {AMT} to (?P<merchant_raw>\S*(?:Dewa|Elife|Etisalat|Sewa|\bDu\b)\S*)'),
        R('card_repayment', rf'payment instructions of {CUR} {AMT} to [X\d*]*\d{{3,}} has been processed'),
        R('transfer_out', rf'funds transfer request of\s+{CUR} {AMT} .*?processed successfully'),
        R('notice', r'funds transfer request of .*? is under process'),
        R('transfer_out', rf'Outward Remittance\s+Debit.*?{CUR} {AMT}'),
        R('transfer_in', rf'(?:Inward Remittance|Cash Deposit)\s+Credit.*?{CUR} {AMT}'),
        R('transfer_in', rf'An amount of {CUR} {AMT} has been credited to your FAB account'),
        R('other_debit', rf'Cheque Debited Cheque No .*? for {CUR} {AMT}'),
    ],
    'ADIB': [
        R('notice', r'Trx\. of [A-Z]{3} ?0\.00 '),                      # card-provisioning checks, zero amount
        R('purchase', rf'^\s*Trx\. of {CUR} ?{AMT} on your (?:card ending|a/c) \*+(?P<card>\d+) at (?P<merchant_raw>.+?)'
                      r'(?: is Approved\.?\s*(?:Avl\.? card bal is )?|\.? Avl Bal is AED )(?P<balance>-?\d[\d,]*\.?\d*)?'),
        R('purchase', rf'^\s*Transaction of {CUR} {AMT} debited from your a/c \*+(?P<card>\d+) at (?P<merchant_raw>.+?)\.? Avl Bal is AED (?P<balance>-?[\d,]*\.?\d*)'),
        R('refund', rf'Trx\. of {CUR} ?{AMT} on your card ending \*+(?P<card>\d+) at (?P<merchant_raw>.+?) is Reversed'),
        R('refund', rf'POS Trxn on your Account No \*+(?P<card>\d+) at (?P<merchant_raw>.+?) in [A-Z]{{2}} on .*? for {CUR} {AMT} is reversed'),
        R('refund', rf'transaction of {CUR} ?{AMT} at (?P<merchant_raw>.+?) on your Covered Card .*? has been refunded'),
        R('cash_withdrawal', rf'ATM Cash Withdrawal for {CUR} {AMT} was debited'),
        R('card_payment_received', rf'your payment of {CUR} ?{AMT} on .*? for card ending'),
        R('card_repayment', rf'{CUR} {AMT} payment for your Covered Card was debited'),
        R('income', rf'(?:Salary|profit) of {CUR} {AMT} was credited'),
        R('transfer_in', rf'{CUR}\.? {AMT} (?:was credited|deposited)'),
        R('transfer_in', rf'for {CUR} {AMT} received for a/c'),
        R('other_debit', rf'{CUR} {AMT} was debited from your account'),   # ADIB gives no merchant for these
    ],
}
NOTICE_RULE = R('notice', r'.*')

def _num(s):
    if s is None:
        return np.nan
    s = str(s).replace(',', '').strip().rstrip('.')
    try:
        return float(s)
    except ValueError:
        return np.nan

# Balance wording across all four banks, used when the matched rule did not capture one itself.
BALANCE_RX = re.compile(
    r'(?:available (?:account )?balance(?: of your account)? is|Avl\.? ?Bal(?:ance)? is|Current balance is|'
    r'Avl bal:|Avl\.? card bal is|Avl\.? ?Cr\.limit(?: is)?|Avl\.Limit is|Available credit limit is now|Your Avl Bal is)'
    r'\s*(?:AED)?\s*(?P<balance>-?\d[\d,]*\.?\d*)', re.IGNORECASE)

def parse_message(msg, bank):
    text = str(msg).replace('\n', ' ').strip()
    for txn_type, rx in COMMON_RULES + BANK_RULES[bank] + [NOTICE_RULE]:
        m = rx.search(text)
        if m:
            g = m.groupdict()
            balance = _num(g.get('balance'))
            if np.isnan(balance) and txn_type not in ('otp', 'declined', 'notice'):
                b = BALANCE_RX.search(text)
                balance = _num(b.group('balance')) if b else np.nan
            return dict(txn_type=txn_type,
                        amount=_num(g.get('amount')),
                        currency=(g.get('currency') or '').upper() or None,
                        card_last4=g.get('card'),
                        merchant_raw=(g.get('merchant_raw') or '').strip() or None,
                        balance_after=balance)

## 5. Parser unit tests

There is one synthetic example per message format. Amounts, cards and balances are made up. If a bank changes its SMS wording, a new example goes here first.

In [ ]:
PARSER_CASES = [
    ('FAB', 'Credit Card Purchase \nCard No XXXX1234 \nAED 37.65 \nNoon Minutes 80038888 AE \n23/03/25 17:23 \nAvailable Balance AED 27274.06\nYour March statement payment due date is 26/03',
     dict(txn_type='purchase', amount=37.65, currency='AED', card_last4='1234', merchant_raw='Noon Minutes 80038888 AE', balance_after=27274.06)),
    ('FAB', 'Credit Card Purchase \nCard No XXXX1234 \nPKR 1500.00 \nCAREEM 923000000000 PK \n01/04/25 10:00 \nAvailable Balance AED 100.00',
     dict(txn_type='purchase', amount=1500.0, currency='PKR')),
    ('FAB', 'Credit Card Purchase \nCard No XXXX1234 \nAED .00 \nVisa Provisioning Service  AE \n01/04/25 10:00 \nAvailable Balance AED 100.00',
     dict(txn_type='notice')),
    ('FAB', 'Dear Customer, Your payment instructions of AED 410.00 to HomeDewa for consumer number ****** has been processed on 01/02/2024 18:00',
     dict(txn_type='bill_payment', amount=410.0, merchant_raw='HomeDewa')),
    ('FAB', 'Dear Customer, Your payment instructions of AED 6,000.00 to XXXX1234 has been processed on 05/02/2024 18:00',
     dict(txn_type='card_repayment', amount=6000.0)),
    ('FAB', 'Dear Customer, Your payment instructions of AED 6,000.00 to 4727********1234 has been processed on 05/02/2024 18:00',
     dict(txn_type='card_repayment', amount=6000.0)),
    ('FAB', 'Inward Remittance \nCredit  \nAccount XXXX1234 \nAED 5,000.00\nValue Date 01/01/2024 \nAvailable Balance AED 6,000.00',
     dict(txn_type='transfer_in', amount=5000.0)),
    ('FAB', 'ATM Cash withdrawal \nDebit \nAccount XXXX1234 \nCard XXXX5678 \nAED 200.00 \n01/01/24 10:00 \nAvailable Balance AED 100.00',
     dict(txn_type='cash_withdrawal', amount=200.0, card_last4='5678')),
    ('FAB', 'Dear Customer, your transaction at Hardees 123 AE has been declined on your credit card ending with XXXX1234.',
     dict(txn_type='declined')),
    # bank, message, expected fields
    ('Liv', 'Purchase of AED 21.50 with Debit Card ending 1234 at TALABAT, DUBAI. Avl Balance is AED 409.99.',
     dict(txn_type='purchase', amount=21.5, currency='AED', card_last4='1234', merchant_raw='TALABAT, DUBAI', balance_after=409.99)),
    ('Liv', 'Purchase of SAR 30.00 with Debit Card ending 1234 at Hungerstation, Riyadh. Avl Balance is AED 1,337.40.',
     dict(txn_type='purchase', amount=30.0, currency='SAR', balance_after=1337.40)),
    ('Liv', 'Purchase amount of AED 4.00 at GOOGLE *TEMPORARY HOLD on your Debit Card ending 1234 has been refunded to your card account. Avl Bal is AED 9.00.',
     dict(txn_type='refund', amount=4.0, merchant_raw='GOOGLE *TEMPORARY HOLD')),
    ('Liv', 'AED 15.00 has been credited to your account no. ****01 Udrive Rent A Car Dubai:AE',
     dict(txn_type='refund', amount=15.0)),
    ('Liv', 'Cash Withdrawal of AED 200.00 with Debit Card ending 1234 at SOUQ SILICON OASIS DXB, DUBAI. Avl Bal is AED 50.00Most Liv. users enjoy going cashless.',
     dict(txn_type='cash_withdrawal', amount=200.0)),
    ('Liv', 'AED 120.00 has been debited from your account no. ****01 ETISALAT NO.-******1 The available balance is AED 5.00.',
     dict(txn_type='bill_payment', amount=120.0, merchant_raw='ETISALAT')),
    ('Liv', 'AED 0.05 has been debited from your account no. 095-XXX39XXX-01 Value Added Tax(VAT) @5%:O00100854170. The available balance is AED 2,087.40.',
     dict(txn_type='bank_fee', amount=0.05)),
    ('Liv', 'AED 50.00 has been debited from your account no. 095-XXX39XXX-01 to Safe with Spend and Save. The available balance is AED 10.00.',
     dict(txn_type='internal_out', amount=50.0, balance_after=10.0)),
    ('Liv', 'AED 25.00 has been debited from your account no. 095-XXX39XXX-01 SOCIAL TRANSFER; LIV REF .-TXFRS1;MSG-******. The available balance is AED 7.39.',
     dict(txn_type='transfer_out', amount=25.0)),
    ('Liv', 'AED 500.00 has been credited to your account no. 095-XXX39XXX-01 SOCIAL TRANSFER; LIV REF .-TXFRS2;MSG-******',
     dict(txn_type='transfer_in', amount=500.0)),
    ('Liv', 'The OTP for your social payment is 123456. You shouldn\'t share it with anyone.',
     dict(txn_type='otp')),
    ('Liv', 'Transaction of AED 10.00 on your Card ending 1234 was declined due to insufficient limit available on your card.',
     dict(txn_type='declined')),
    ('Mashreq', 'Thank you for using NEO VISA Debit Card Card ending 1234 for AED 25.00 at ROAD & TRANSPORT on 27-SEP-2023 07:55 AM. Available Balance is AED 1,000.00',
     dict(txn_type='purchase', amount=25.0, merchant_raw='ROAD & TRANSPORT', balance_after=1000.0)),
    ('Mashreq', 'Thank you for using NEO VISA Debit Card Card ending 1234 for USD 16.00 at Epidemic Sound AB on 01-JAN-2024 10:00 AM. Available Balance is AED 900.00',
     dict(txn_type='purchase', amount=16.0, currency='USD')),
    ('Mashreq', 'Mashreq Debit Card ending 1234 was used for a transaction of AED 9.99 at GOOGLE*SNAPCHAT on 01-FEB-2024 09:00 AM.',
     dict(txn_type='purchase', amount=9.99, merchant_raw='GOOGLE*SNAPCHAT')),
    ('Mashreq', 'Your Card ending with 1234 was used for cash withdrawal of AED 100.00 at 1-SOUK EXTRA M, DUBAI on 01-APR-2024 10:00 AM. Avl bal: AED 50.00',
     dict(txn_type='cash_withdrawal', amount=100.0)),
    ('Mashreq', 'Your AC No: XXXXXXXX1234 is credited with AED 126.00 as Visa Refund. Login to Online Banking for details.',
     dict(txn_type='refund', amount=126.0)),
    ('Mashreq', 'Your AC No: XXXXXXXX1234 is debited with AED 26.25 for Fall Below Fees. Login to Online Banking for details.',
     dict(txn_type='bank_fee', amount=26.25)),
    ('Mashreq', 'Your AC No:XXXXXXXX1234 is debited with AED 300.00 for Aani Instant Payments (Local IPP Transfer). Login to Online Banking for details',
     dict(txn_type='transfer_out', amount=300.0)),
    ('Mashreq', 'Your AC No: XXXXXXXX1234 is credited with AED 5,000.00 for Salary. Login to Online Banking for details.',
     dict(txn_type='income', amount=5000.0)),
    ('ADCB', 'Your Cr.Card ***1234 was used for AED65.00 on 24/06/2023 22:16:10 at ZEN RESTAURANT,DUBAI-AE. Avl. Cr.limit is AED1753.16',
     dict(txn_type='purchase', amount=65.0, currency='AED', merchant_raw='ZEN RESTAURANT,DUBAI-AE', balance_after=1753.16)),
    ('ADCB', 'Credit Card ***1234 used for AED273.99 (+2.1% foreign txn fee) on 06/06/2025 23:04:16 at Spotify,Stockholm-SE. Avl. Cr.limit AED5311.47',
     dict(txn_type='purchase', amount=273.99, merchant_raw='Spotify,Stockholm-SE')),
    ('ADCB', 'Your Cr.Card ***1234 was used for USD20.00 on 01/02/2024 10:00:00 at hostinger.com,Larnaka-CY. Avl. Cr.limit is AED100.00',
     dict(txn_type='purchase', amount=20.0, currency='USD')),
    ('ADCB', 'Congratulations on your first 3D secure transaction. Your Credit Card *** 1234 was used for AED 35.49 on 12/03/2022 23:52:28 at Amazon.ae,Dubai- AE. Available credit limit is now AED 2964.51.',
     dict(txn_type='purchase', amount=35.49, balance_after=2964.51)),
    ('ADCB', 'An amount of AED10.00 has been reversed to your Credit Card ***1234 on 01/01/2024 10:00:00 by Vox Cinemas,Dubai-AE.',
     dict(txn_type='refund', amount=10.0, merchant_raw='Vox Cinemas,Dubai-AE')),
    ('ADCB', 'AED250.00 debited from Acc/Cr.Card ***1234 for DEWA on 01-02-2024 10:00:00 through ADCB Mobile App.Avl.Limit is AED 100',
     dict(txn_type='bill_payment', amount=250.0, merchant_raw='DEWA')),
    ('ADCB', 'Your payment of AED 3000 against Credit Card no. ***1234 was received at 10:00 AM on 01/02/2024. Thank you.',
     dict(txn_type='card_payment_received', amount=3000.0)),
    ('ADCB', 'Cr.Card ***1234 Billing alert: Total due to avoid fin. charges: AED1500.00. Due date Jun 10 2024; Pay min. AED100.00 by due date.',
     dict(txn_type='notice')),
    ('ADIB', 'Trx. of AED45.00 on your card ending *510 at GLOBAL VILLAGE D, UAE is Approved. Avl. card bal is 1955.00. Trx Date: 28/01/23 18:54',
     dict(txn_type='purchase', amount=45.0, card_last4='510', merchant_raw='GLOBAL VILLAGE D, UAE', balance_after=1955.0)),
    ('ADIB', 'Trx. of AED 451.20 on your card ending **0121 at LuluHypermarket KHALID in Abu Dhabi, UNITED ARAB EMI is Approved. Avl card bal is 15648.80.Trx Date: 29/07/20 19:29.',
     dict(txn_type='purchase', amount=451.2, merchant_raw='LuluHypermarket KHALID in Abu Dhabi, UNITED ARAB EMI', balance_after=15648.80)),
    ('ADIB', 'Trx. of AED 50.00 on your a/c ****1234 at ABU DHABI NATIONAL OIL ABU DHABI AE. Avl Bal is AED 12956.50',
     dict(txn_type='purchase', amount=50.0, balance_after=12956.50)),
    ('ADIB', 'Transaction of AED 104.00 debited from your a/c ****1234 at Kuwait Food Co Americana 600522252 AE. Avl Bal is AED 579.68',
     dict(txn_type='purchase', amount=104.0, balance_after=579.68)),
    ('ADIB', 'Trx. of USD0.00 on your card ending *944 at Visa Provisionin, UAE is Approved. Trx Date: 08/07/23 22:29',
     dict(txn_type='notice')),
    ('ADIB', 'Trx. of AED10.00 on your card ending *944 at Noon, UAE is Reversed. Trx Date: 01/01/23 10:00',
     dict(txn_type='refund', amount=10.0)),
    ('ADIB', 'Dear Customer, AED 300.00 was debited from your account ****1234. Your available account balance is AED 100.00',
     dict(txn_type='other_debit', amount=300.0, balance_after=100.0)),
    ('ADIB', 'Dear Customer, your payment of AED 100.00 on 7/29/2020 for card ending with **1234 has been credited. Thank you.',
     dict(txn_type='card_payment_received', amount=100.0)),
    ('ADIB', 'Trx. for Covered Card ending *944 at TALABAT, UAE on 01/01/23 10:00 for AED50.00 has been rejected due to insufficient balance',
     dict(txn_type='declined')),
]

failures = []
for bank, msg, expected in PARSER_CASES:
    got = parse_message(msg, bank)
    for k, v in expected.items():
        g = got[k]
        ok = (abs(g - v) < 1e-9) if isinstance(v, float) else (g == v)
        if not ok:
            failures.append(f'{bank}: {k} expected {v!r} got {g!r} | {msg[:60]}')
check(f'parser unit tests ({len(PARSER_CASES)} cases)', not failures, '; '.join(failures))

## 6. Parse every message

In [ ]:
parsed = pd.DataFrame([parse_message(m, b) for m, b in zip(raw.message, raw.bank)])
tx = pd.concat([raw, parsed], axis=1)
pd.crosstab(tx.txn_type, [tx.dataset, tx.bank], margins=True)

## 7. Clean and enrich

- **Merchant and location.** Each bank writes the location differently (`TALABAT, DUBAI`, `ZEN RESTAURANT,DUBAI-AE`, `X in Abu Dhabi, UNITED ARAB EMI`, `X ABU DHABI AE`). This step splits it off.
- **`merchant_clean`** is a normalised name. A small brand table merges the obvious variants (e.g. 7 spellings of Talabat). The table can be extended as needed.
- **Money-transfer merchants** (exchange houses, Western Union, investment top-ups) charged to a card are reclassified as `transfer_out`. **BNPL** merchants (Tabby, Tamara, Talabat Pay Later/Postpaid) stay as purchases but get the `is_bnpl` flag, because the real item isn't visible.
- **Currency.** `amount_aed` converts pegged currencies at fixed rates; other currencies stay NaN and keep their original `amount`/`currency`.
- **Duplicates.** An identical message from the same user and bank is a duplicate SMS. Real transactions differ at least in balance, so the first copy is kept and later ones are flagged.

In [ ]:
UAE_CITIES = r'(?:ABU ?DHABI|DUBAI|SHARJAH|SHJ|AJMAN|AL AIN|RAS AL KHAIMA|FUJAIRAH|UMM AL QUWAIN|DXB|AUH)'

def split_merchant(raw_m, bank):
    # Return (merchant, location, country) from the raw merchant text.
    if not isinstance(raw_m, str) or not raw_m.strip():
        return None, None, None
    s = re.sub(r'\s+', ' ', raw_m).strip()
    if bank == 'ADCB':                                  # NAME,CITY-CC
        m = re.match(r'(.*),\s*(.*?)-\s*([A-Z]{2})$', s)
        if m:
            return m.group(1).strip(), m.group(2).strip() or None, m.group(3)
        return s, None, None
    if bank == 'Liv':                                   # NAME, CITY
        if ', ' in s:
            name, loc = s.rsplit(', ', 1)
            return name.strip(), loc.strip(), None
        return s, None, None
    if bank == 'FAB':                                   # NAME [PHONE] [CITY] CC
        m = re.match(r'(.*?)\s+([A-Z]{2})$', s)
        name, cc = (m.group(1).strip(), m.group(2)) if m else (s, None)
        loc = re.search(r'\s+((?:ABU ?DHABI|DUBAI|SHARJAH|AJMAN|AL AIN|RAS AL KHAIMAH|FUJAIRAH|RIYADH|JEDDAH|KARACHI|LAHORE|LONDON|CAIRO|[A-Z][a-z]+))$', name, re.I)
        if loc and loc.start() > 2:
            return name[:loc.start()].strip(), loc.group(1), cc
        return name, None, cc
    if bank == 'ADIB':
        m = re.match(r'(.*?) in (.*?), ?(UNITED ARAB EMI|UNITED STATES|.*)$', s)   # NAME in CITY, COUNTRY
        if m:
            return m.group(1).strip(), m.group(2).strip() or None, 'AE' if 'ARAB' in m.group(3) else None
        m = re.match(r'(.*), ?(UAE|UNITED STATES|IRELAND|.*)$', s)                # NAME, COUNTRY
        if m:
            return m.group(1).strip(), None, 'AE' if m.group(2) == 'UAE' else None
        m = re.match(rf'(.*?)\s+{UAE_CITIES}?\s*([A-Z]{{2}})$', s)              # NAME CITY CC
        if m:
            loc = re.search(rf'{UAE_CITIES}\s*[A-Z]{{2}}$', s)
            return m.group(1).strip(), loc.group(0)[:-2].strip() if loc else None, m.group(2)
        return s, None, None
    return s, None, None                                # Mashreq: no location in the text

# Ordered (pattern, canonical name). First match wins.
BRANDS = [
    (r'TALABAT|DELIVERY HERO', 'Talabat'), (r'KEETA', 'Keeta'), (r'NOON ?FOOD', 'Noon Food'),
    (r'NOON ?MINUTES', 'Noon Minutes'), (r'NOON ?DAILY', 'Noon Daily'), (r'NOON ?ONE', 'Noon One'),
    (r'\bNOON\b|NOON\.COM|NOON ?PAY', 'Noon'), (r'AMAZON', 'Amazon'), (r'CAREEM', 'Careem'),
    (r'UBER|UBR\*', 'Uber'), (r'BOLT\.EU|\bBOLT\b', 'Bolt'), (r'SNAPCHAT', 'Snapchat'), (r'ZOMATO', 'Zomato'),
    (r'DELIVEROO', 'Deliveroo'), (r'INSTASHOP', 'Instashop'), (r'STEAM', 'Steam'), (r'SPOTIFY', 'Spotify'),
    (r'NETFLIX', 'Netflix'), (r'APPLE\.COM|ITUNES|APPLE ONLINE', 'Apple'), (r'PLAYSTATION|SONY INTERACTIVE', 'PlayStation'),
    (r'GOOGLE \*TEMPORARY HOLD|GOOGLE\*TEMPORARY', 'Google (temporary hold)'), (r'GOOGLE', 'Google'),
    # Restaurant brands come before fuel brands: 'MCDONALDS-ADNOC 74' is a McDonald's inside a petrol station, not fuel.
    (r'MCDONALD', "McDonald's"), (r'\bKFC\b', 'KFC'), (r'PIZZA HUT', 'Pizza Hut'), (r'DOMINO', "Domino's"),
    (r'BURGER KING', 'Burger King'), (r'STARBUCKS', 'Starbucks'), (r'KUWAIT FOOD CO|AMERICANA', 'Americana'),
    (r'\bADNOC\b|ABU DHABI NATIONAL OIL', 'ADNOC'), (r'\bENOC\b|EPPCO', 'ENOC'), (r'EMARAT', 'Emarat'),
    (r'CARREFOUR', 'Carrefour'), (r'LULU', 'Lulu'), (r'\bVIVA\b', 'Viva'), (r'SPINNEYS', 'Spinneys'),
    (r'UNION COOP', 'Union Coop'), (r'CITY ?FARM', 'City Farms'), (r'SOUQ EXTRA|SOUK EXTRA', 'Souq Extra'),
    (r'\bRTA\b|SALIK|NOL ', 'RTA'), (r'SMART ?D(?:UBAI|XB) ?GOV|DUBAI ?E?GOV', 'Dubai Government'),
    (r'\bDEWA\b|DUBAI ELECTRICITY', 'DEWA'), (r'\bSEWA\b', 'SEWA'), (r'\bFEWA\b', 'FEWA'),
    (r'ETISALAT|\bE&\b', 'Etisalat'), (r'^DU\b|\bDU (?:QUICK|ONE|GOOGLE)|GSM SERVICE', 'du'),
    (r'TABBY', 'Tabby'), (r'TAMARA', 'Tamara'), (r'VOX|MAJID AL FUTTAIM CIN', 'VOX Cinemas'),
    (r'FACEBK|FACEBOOK', 'Facebook Ads'), (r'CHOITH?RAM', 'Choithrams'), (r'TIKTOK', 'TikTok'),
    (r'PATREON', 'Patreon'), (r'UPWORK', 'Upwork'), (r'PAYPAL', 'PayPal'), (r'ZIINA', 'Ziina'),
]
BRANDS = [(re.compile(p, re.I), name) for p, name in BRANDS]
FUEL_BRANDS = {'ADNOC', 'ENOC', 'Emarat'}
FOOD_AT_STATION = re.compile(r'CAFE|COFFEE|TIM HORTON|COSTA|CARIBOU|DUNKIN|SUBWAY|HARDEE|POPEYE|BURGER|PIZZA|SHAWARMA|BAKERY|JUICE|RESTAURANT|KITCHEN|TEA\b|KARAK|CHICKEN|DONUT', re.I)

TRANSFER_MERCHANTS = re.compile(r'EXCHAN|WU\.COM|WESTERN UNION|SEND_MONEY|SARWA|CAREEM PAY TOPUP|REMIT', re.I)
BNPL_MERCHANTS = re.compile(r'TABBY|TAMARA|PAY LATER|POSTPAID|POSTPAY', re.I)

def clean_name(name):
    if not isinstance(name, str) or not name.strip():
        return None
    for rx, canon in BRANDS:
        if rx.search(name):
            if canon in FUEL_BRANDS and FOOD_AT_STATION.search(name):
                continue                               # cafe/restaurant tenant at a petrol station: keep its own name
            return canon
    s = re.sub(r'^(?:SP|FAT|WL|TAP|MYF|SQ|SUMUP|NOMOD)\s*\*\s*', '', name, flags=re.I)   # payment-processor prefixes
    s = re.sub(r'\+?\d[\d -]{5,}', ' ', s)            # phone numbers and long reference digits
    s = re.sub(r'\b\d+\b', ' ', s)                     # store numbers
    s = re.sub(r'[*#/]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip(' -,.')
    return s.upper() or None

df = tx.copy()
split = [split_merchant(m, b) for m, b in zip(df.merchant_raw, df.bank)]
df['merchant'] = [x[0] for x in split]
df['location'] = [x[1] for x in split]
df['country'] = [x[2] for x in split]
df['merchant_clean'] = df.merchant.map(clean_name)
# A few real purchases carry no usable merchant text (e.g. 'at , Dubai' or only a phone number).
no_name = df.txn_type.isin({'purchase', 'bill_payment'}) & df.merchant_clean.isna()
df.loc[no_name, 'merchant_clean'] = 'UNKNOWN MERCHANT'

# Card charges that are really money transfers
is_xfer = df.txn_type.eq('purchase') & df.merchant_raw.fillna('').str.contains(TRANSFER_MERCHANTS)
df.loc[is_xfer, 'txn_type'] = 'transfer_out'
print(f'{is_xfer.sum()} card charges reclassified as transfer_out:',
      df.loc[is_xfer, 'merchant_clean'].value_counts().to_dict())

df['is_bnpl'] = df.txn_type.eq('purchase') & df.merchant_raw.fillna('').str.contains(BNPL_MERCHANTS)

# Currency
df['amount_aed'] = df.amount * df.currency.map(AED_PER_UNIT)
df['is_foreign'] = df.currency.notna() & (df.currency.ne('AED') | (df.country.notna() & df.country.ne('AE')))

# Direction and signed amount
DEBIT = {'purchase', 'bill_payment', 'cash_withdrawal', 'bank_fee', 'transfer_out', 'card_repayment',
         'loan_repayment', 'internal_out', 'other_debit'}
CREDIT = {'refund', 'transfer_in', 'income', 'internal_in', 'card_payment_received'}
df['direction'] = np.select([df.txn_type.isin(DEBIT), df.txn_type.isin(CREDIT)], ['debit', 'credit'], 'none')
df['signed_amount_aed'] = np.where(df.direction.eq('debit'), -df.amount_aed,
                                   np.where(df.direction.eq('credit'), df.amount_aed, np.nan))

# Duplicates, cutoff, user aliases
df = df.sort_values(['dataset', 'user', 'bank', 'date', 'source_row']).reset_index(drop=True)
df['is_duplicate'] = df.duplicated(['dataset', 'user', 'bank', 'message'], keep='first')
before = len(df)
df = df[~(df.dataset.eq('personal') & (df.date > CUTOFF))].reset_index(drop=True)
print(f'{before - len(df)} rows after {CUTOFF.date()} removed')
df['user'] = df.user.replace(USER_ALIASES)

# Expense flags
df['is_expense'] = df.txn_type.isin({'purchase', 'bill_payment'}) & ~df.is_duplicate
df['is_borderline'] = df.txn_type.isin({'refund', 'cash_withdrawal', 'bank_fee'}) & ~df.is_duplicate

# Empty column for your own 10-class labels
df['category'] = pd.NA

df['txn_id'] = (df.dataset.str[0] + '-' + df.user.astype(str) + '-' + df.bank + '-'
                + df.groupby(['dataset', 'user', 'bank']).cumcount().add(1).astype(str).str.zfill(5))

COLUMNS = ['txn_id', 'dataset', 'user', 'bank', 'date', 'txn_type', 'direction', 'is_expense', 'is_borderline',
           'is_duplicate', 'is_bnpl', 'is_foreign', 'amount', 'currency', 'amount_aed', 'signed_amount_aed',
           'merchant_clean', 'merchant', 'location', 'country', 'card_last4', 'balance_after', 'category',
           'message', 'source_file', 'source_sheet', 'source_row']
df = df[COLUMNS]
df[df.is_expense].sample(8, random_state=1)[['bank', 'date', 'amount', 'currency', 'merchant_clean', 'merchant', 'location']]

## 8. Data-quality tests

In [ ]:
exp = df[df.is_expense]
money_types = ['purchase', 'bill_payment', 'refund', 'cash_withdrawal', 'bank_fee', 'transfer_out', 'transfer_in',
               'income', 'card_payment_received', 'card_repayment', 'internal_out', 'internal_in', 'other_debit']

check('no missing date / bank / user / message',
      df[['date', 'bank', 'user', 'message']].notna().all().all())
check('txn_id is unique', df.txn_id.is_unique)
check('every money-moving row has a positive amount and a 3-letter currency',
      (df[df.txn_type.isin(money_types)].amount.gt(0) & df[df.txn_type.isin(money_types)].currency.str.fullmatch('[A-Z]{3}')).all(),
      df[df.txn_type.isin(money_types) & ~(df.amount.gt(0))].message.head(3).tolist())
check('every expense has a merchant_clean value', exp.merchant_clean.notna().all(),
      exp[exp.merchant_clean.isna()].message.head(3).tolist())
unknown = exp.merchant_clean.eq('UNKNOWN MERCHANT')
check('under 1% of expenses have an unknown merchant', unknown.mean() < 0.01,
      f'{unknown.sum()} rows: ' + str(exp[unknown].message.head(3).tolist()), hard=False)
check(f'personal dataset ends on or before {CUTOFF.date()}',
      df[df.dataset.eq('personal')].date.max() <= CUTOFF, df[df.dataset.eq('personal')].date.max())
check('no duplicate messages left among expenses',
      not exp.duplicated(['user', 'bank', 'message']).any())
check('expenses only contain purchase / bill_payment', set(exp.txn_type) <= {'purchase', 'bill_payment'})
check('pegged currencies all converted to AED',
      df[df.currency.isin(AED_PER_UNIT.keys())].amount_aed.notna().all())

# No missed transactions: a "notice" row should not describe money actually moving.
MOVED = r'has been debited|was debited|Purchase of|was used for AED|was used for [A-Z]{3}\d|Cash Withdrawal of'
NOTICE_OK = r'will be debited|were not debited|bonus TouchPoints|upgrading our systems|may be unavailable'
missed = df[df.txn_type.eq('notice') & df.message.str.contains(MOVED, case=False)
            & ~df.message.str.contains(NOTICE_OK, case=False)]
check('no transaction left unparsed as a notice', missed.empty, missed.message.head(3).tolist())

# Amounts that look implausible for a single card purchase (soft: just for review)
big = exp[exp.amount_aed > 20000]
check('no single expense above AED 20,000', big.empty,
      big[['user', 'bank', 'amount_aed', 'merchant_clean']].to_dict('records'), hard=False)

# Balance chain: for consecutive messages that both show a balance, previous balance +/- this amount
# should equal this balance. A mismatch nearly always means an SMS in between is missing (or carried no
# balance, e.g. transfers and refunds on Liv), not a parsing error, so this is reported as a coverage metric.
# On credit cards the "balance" is the available limit, which works the same way.
# Liv scores lowest for two known bank-side reasons, checked by hand: its 'Avl Balance' often lags a purchase
# (same balance on consecutive SMS), and foreign/online purchases take an extra ~3% fee not shown in the SMS.
def balance_consistency(g):
    g = g[g.balance_after.notna() & g.signed_amount_aed.notna()].sort_values('date')
    prev = g.balance_after.shift()
    ok = ((prev + g.signed_amount_aed) - g.balance_after).abs() < 0.02
    return pd.Series({'pairs': int(prev.notna().sum()), 'agree': ok[prev.notna()].mean()})

bal = (df[~df.is_duplicate].groupby(['dataset', 'user', 'bank'])
         .apply(balance_consistency, include_groups=False))
bal = bal[bal.pairs >= 10]
print('\nBalance-chain agreement by account:')
print(bal.assign(agree=bal.agree.map('{:.0%}'.format)).to_string())

# Parser sanity: where the chain DOES line up, the parsed amount is exactly right. Check that the biggest
# accounts reach a reasonable level so a systematic amount bug (e.g. dropped thousands separator) would show.
check('balance chain agrees for at least 40% of pairs on every account with 10+ pairs',
      (bal.agree >= 0.4).all(), bal.agree.round(2).to_dict(), hard=False)

## 9. Privacy tests

These run on the final table. They check for things the censoring should already have removed: names from `BANNED_TERMS`, personal greetings, transfer recipients, unmasked transfer memos, email addresses and full account numbers.

In [ ]:
msgs = df.message

hits = {t: int(msgs.str.contains(re.escape(t), case=False).sum()) for t in BANNED_TERMS}
hits = {t: n for t, n in hits.items() if n}
# 'SHAH' as part of merchant names (Al Shahama etc.) is fine; only flag the full banned terms.
check('no banned names anywhere', not hits, hits)

greet = msgs.str.extract(r'\b(?:Dear|Hi|Hello) ([A-Z][a-z]{2,})\b')[0].dropna()
greet = greet[~greet.isin(['Customer', 'Customers', 'Cardholder', 'Valued', 'There', 'there', 'FAB'])]
check('no personal greetings (Dear <name>)', greet.empty, greet.value_counts().head().to_dict())

recip = msgs[msgs.str.contains(r'transfer (?:funds )?(?:of )?(?:AED [\d.,]+ )?to (?!\*|your|other|Sarwa|.*account)[A-Z][a-z]+', regex=True)]
check('no named transfer recipients', recip.empty, recip.head(3).tolist())

memo = msgs.str.extract(r';MSG-([^.\n]*)')[0].dropna()
memo = memo[~memo.str.fullmatch(r'\**')]
check('all transfer memos masked', memo.empty, memo.head(5).tolist())

emails = msgs.str.extract(r'([\w.+-]+@[\w-]+(?:\.[A-Za-z]{2,})+)')[0].dropna()
emails = emails[~emails.str.contains(r'@(?:liv\.me|google\.com|mashreq\.com|adcb\.com|adib\.ae)$', case=False)]
check('no personal email addresses', emails.empty, emails.unique()[:5].tolist())

acct = msgs[msgs.str.contains(r'\b(?:account|a/c|acc)\.?(?: no\.?| number)?:? ?\d{7,}', case=False)]
check('no full account numbers', acct.empty, acct.head(3).tolist())

## 10. Summary

In [ ]:
summary = (df.groupby(['dataset', 'user', 'bank'])
             .agg(messages=('txn_id', 'size'),
                  expenses=('is_expense', 'sum'),
                  borderline=('is_borderline', 'sum'),
                  duplicates=('is_duplicate', 'sum'),
                  bnpl=('is_bnpl', 'sum'),
                  foreign=('is_foreign', lambda s: int((s & df.loc[s.index, 'is_expense']).sum())),
                  expense_aed=('amount_aed', lambda s: s[df.loc[s.index, 'is_expense']].sum().round(2)),
                  first=('date', 'min'), last=('date', 'max'))
             .reset_index())
display(summary)
print('Usable expenses:', summary.groupby('dataset').expenses.sum().to_dict())

print('\nTop merchants (personal):')
display(df[df.is_expense & df.dataset.eq('personal')].merchant_clean.value_counts().head(15))

pd.crosstab(df.txn_type, df.dataset)

In [ ]:
# Expected warning on the Khalifa data: user2's credit card (..121) also appears in user1/3/4's messages and
# user6's card (..510) once in user2's. They aren't copies, so the users likely share a household or phone.
# Card numbers seen in more than one user's messages: a sign two "users" share a card, phone or household.
# (ADIB writes card endings as *121 or **0121, so only the last 3 digits are compared.)
cards = (df[df.card_last4.notna()].assign(card4=df.card_last4.str[-3:])
           .groupby(['dataset', 'card4']).user.unique())
shared = cards[cards.map(len) > 1]
check('no card shared between users within a dataset', shared.empty,
      {f'{d} card ..{c}': list(u) for (d, c), u in shared.items()}, hard=False)

## 11. Test report and export

In [ ]:
report = pd.DataFrame(TEST_RESULTS)
display(report)
hard_fail = report[report.hard & ~report.passed]
assert hard_fail.empty, f'{len(hard_fail)} hard test(s) failed - fix before exporting:\n{hard_fail.test.tolist()}'
print(f"All {int(report.hard.sum())} hard tests passed; {int((~report.hard & ~report.passed).sum())} warning(s).")

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

def export(dataset, stem):
    d = df[df.dataset.eq(dataset)].drop(columns='dataset')
    path = OUTPUT_DIR / f'{stem}.xlsx'
    with pd.ExcelWriter(path, engine='openpyxl') as xw:
        d[d.is_expense].to_excel(xw, sheet_name='expenses', index=False)
        d.to_excel(xw, sheet_name='all_messages', index=False)
        summary[summary.dataset.eq(dataset)].to_excel(xw, sheet_name='summary', index=False)
        report.to_excel(xw, sheet_name='tests', index=False)
    d.to_csv(OUTPUT_DIR / f'{stem}_all.csv', index=False)
    d[d.is_expense].to_csv(OUTPUT_DIR / f'{stem}_expenses.csv', index=False)
    print(f'{path}: {int(d.is_expense.sum()):,} expenses / {len(d):,} messages')

export('personal', 'personal_transactions')
export('khalifa', 'khalifa_transactions')     # kept separate on purpose